# Week 9, Exercise 8: Coding an Encoder-Only Transformer

We implement a small BERT-style classifier, which prepends a `[CLS]` token and uses unmasked self-attention, and train it to decide whether a sequence of 8 digits is sorted.

This notebook contains the full solution of Exercise 8 from the [Week 9 optional exercises](https://utm-csc413.github.io/2026F-website/optional_exercises/w09/exercises_w09.html). The saved outputs come from a run on a CPU; your numbers may differ slightly on Colab or with other PyTorch versions. Each training run takes a few seconds to about half a minute on a CPU.

## Building blocks from Exercise 7

`MultiHeadAttention`, `causal_mask`, and `Block`, copied from Exercise 7.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):                       # (B, n, d_model) -> (B, h, n, d_k)
        B, n, _ = x.shape
        return x.view(B, n, self.h, self.d_k).transpose(1, 2)

    def forward(self, x_q, x_kv, mask=None):
        Q = self.split_heads(self.W_q(x_q))         # (B, h, n_q, d_k)
        K = self.split_heads(self.W_k(x_kv))        # (B, h, n_k, d_k)
        V = self.split_heads(self.W_v(x_kv))        # (B, h, n_k, d_k)
        scores = Q @ K.transpose(-2, -1) / self.d_k ** 0.5   # (B, h, n_q, n_k)
        if mask is not None:
            scores = scores + mask                  # mask: (n_q, n_k), 0 or -inf
        C = torch.softmax(scores, dim=-1) @ V       # (B, h, n_q, d_k)
        B, _, n_q, _ = C.shape
        C = C.transpose(1, 2).reshape(B, n_q, self.h * self.d_k)  # concatenate heads
        return self.W_o(C)

def causal_mask(n):
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)

class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        a = self.norm1(x + self.attn(x, x, mask))   # self-attention, Add & Norm
        return self.norm2(a + self.ffn(a))          # FFN, Add & Norm

## Data

Only about 0.02% of random digit sequences are sorted, so `make_batch` builds a balanced dataset: each positive is a sorted random sequence, and each negative is a shuffle of random digits that is not sorted.

In [2]:
def make_batch(batch_size, n=8):
    x = torch.randint(10, (batch_size, n))
    y = torch.randint(2, (batch_size,))              # 1 = sorted, 0 = not sorted
    for i in range(batch_size):
        if y[i] == 1:
            x[i] = x[i].sort().values
        else:
            while (x[i][1:] >= x[i][:-1]).all():     # redraw until not sorted
                if (x[i] == x[i][0]).all():          # all digits equal: every order is sorted
                    x[i] = torch.randint(10, (n,))
                else:
                    x[i] = x[i][torch.randperm(n)]
    return x, y

## (a) The encoder-only classifier

In [3]:
class EncoderClassifier(nn.Module):
    def __init__(self, vocab_size=10, max_len=8, d_model=64, n_heads=4, d_ff=128,
                 n_layers=2, use_pos=True):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_emb = nn.Embedding(max_len + 1, d_model) if use_pos else None
        self.blocks = nn.ModuleList([Block(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.head = nn.Linear(d_model, 2)

    def forward(self, idx):                           # idx: (B, n)
        B, n = idx.shape
        x = torch.cat([self.cls.expand(B, 1, -1), self.tok_emb(idx)], dim=1)  # (B, n+1, d)
        if self.pos_emb is not None:
            x = x + self.pos_emb(torch.arange(n + 1))
        for block in self.blocks:
            x = block(x)                              # no mask: every token sees every token
        return self.head(x[:, 0])                     # logits from the [CLS] position

## (b) Training with position embeddings

In [4]:
def train(use_pos=True, steps=1000):
    torch.manual_seed(0)
    model = EncoderClassifier(use_pos=use_pos)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for step in range(steps):
        x, y = make_batch(64)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    x_test, y_test = make_batch(2000)
    with torch.no_grad():
        acc = (model(x_test).argmax(dim=1) == y_test).float().mean()
    print(f"use_pos={use_pos}: final loss {loss.item():.4f}, test accuracy {acc:.3f}")
    return model

train(use_pos=True)

use_pos=True: final loss 0.0051, test accuracy 0.999


EncoderClassifier(
  (tok_emb): Embedding(10, 64)
  (pos_emb): Embedding(9, 64)
  (blocks): ModuleList(
    (0-1): 2 x Block(
      (attn): MultiHeadAttention(
        (W_q): Linear(in_features=64, out_features=64, bias=True)
        (W_k): Linear(in_features=64, out_features=64, bias=True)
        (W_v): Linear(in_features=64, out_features=64, bias=True)
        (W_o): Linear(in_features=64, out_features=64, bias=True)
      )
      (ffn): Sequential(
        (0): Linear(in_features=64, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=64, bias=True)
      )
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (head): Linear(in_features=64, out_features=2, bias=True)
)

## (c) Training without position embeddings

In [5]:
train(use_pos=False)

use_pos=False: final loss 0.6932, test accuracy 0.503


EncoderClassifier(
  (tok_emb): Embedding(10, 64)
  (blocks): ModuleList(
    (0-1): 2 x Block(
      (attn): MultiHeadAttention(
        (W_q): Linear(in_features=64, out_features=64, bias=True)
        (W_k): Linear(in_features=64, out_features=64, bias=True)
        (W_v): Linear(in_features=64, out_features=64, bias=True)
        (W_o): Linear(in_features=64, out_features=64, bias=True)
      )
      (ffn): Sequential(
        (0): Linear(in_features=64, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=64, bias=True)
      )
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (head): Linear(in_features=64, out_features=2, bias=True)
)

Without position embeddings, the model does no better than chance, and the loss stays at $\log 2 \approx 0.693$. By Exercise 5, unmasked self-attention without positional information is permutation equivariant, and the FFN, residual connections, and LayerNorm all act on each token separately. Shuffling the 8 digit tokens therefore only shuffles their outputs, and the `[CLS]` output (which attends to all of them) does not change at all. The model only sees the *multiset* of digits. Any multiset of digits can appear both sorted (as a positive) and shuffled (as a negative), so the multiset carries essentially no information about the label. With position embeddings, each token knows where it is, and the task becomes easy.